In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("superstore.csv")

In [10]:
df = pd.DataFrame(data)

In [35]:
df.head()

,Category,City,Country,Customer.ID,Customer.Name,Discount,Market,记录数,Order.Date,Order.ID,...,Sales,Segment,Ship.Date,Ship.Mode,Shipping.Cost,State,Sub.Category,Year,Market2,weeknum
0,Office Supplies,Los Angeles,United States,LS-172304,Lycoris Saunders,0.0,US,1,2011-01-07 00:00:00.000,CA-2011-130813,...,19,Consumer,2011-01-09 00:00:00.000,Second Class,4.37,California,Paper,2011,North America,2
1,Office Supplies,Los Angeles,United States,MV-174854,Mark Van Huff,0.0,US,1,2011-01-21 00:00:00.000,CA-2011-148614,...,19,Consumer,2011-01-26 00:00:00.000,Standard Class,0.94,California,Paper,2011,North America,4
2,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,21,Consumer,2011-08-09 00:00:00.000,Standard Class,1.81,California,Paper,2011,North America,32
3,Office Supplies,Los Angeles,United States,CS-121304,Chad Sievert,0.0,US,1,2011-08-05 00:00:00.000,CA-2011-118962,...,111,Consumer,2011-08-09 00:00:00.000,Standard Class,4.59,California,Paper,2011,North America,32
4,Office Supplies,Los Angeles,United States,AP-109154,Arthur Prichep,0.0,US,1,2011-09-29 00:00:00.000,CA-2011-146969,...,6,Consumer,2011-10-03 00:00:00.000,Standard Class,1.32,California,Paper,2011,North America,40


#### Nettoyage des chaines de caractères (suppression des espaces invisibles)

In [34]:
df["Customer.ID"] = df['Customer.ID'].str.strip()
df['Customer.Name'] = df['Customer.Name'].str.strip()
df["Product.Name"] = df['Product.Name'].str.strip()

#### Standardisation des dates . On force le format date pour pouvoir faire des analyses chronologiques

In [37]:
df["Order.Date"] = pd.to_datetime(df["Order.Date"], errors='coerce')
df['Ship.Date'] = pd.to_datetime(df["Ship.Date"],errors= "coerce")

#### Création de 3 tables distinctes selon le modèle relationnel et supprimons les doublons

In [ ]:
customers = df[["Customer.ID","Customer.Name","Segment","Country","City","State","Region"]].copy()
customers.drop_duplicates("Customer.ID",inplace=True)


In [25]:
products = df[["Product.ID","Category","Sub.Category","Product.Name"]].copy()
products.drop_duplicates("Product.ID",inplace=True)

In [26]:
transactions = df[["Order.ID","Order.Date","Ship.Date","Ship.Mode","Customer.ID","Product.ID","Sales","Quantity","Discount","Profit"]].copy()
transactions.drop_duplicates("Order.ID",inplace=True)

#### Création du "Flag" d'audit interne (Alerte sur les grosses remises à perte)
lorsque le profit est négatif + la remise dépasse 40%

In [ ]:
transactions['Red.Flag'] = (transactions['Profit'] < 0) & (transactions['Discount'] > 0.4)
transactions[transactions["Red.Flag"]==True]

#### Calcul du coût de revient et vérifions s'il ya des transactions avec un coût de revient négatif ou CA négatif

In [51]:
transactions["Cout de Revient"] = transactions["Sales"] - transactions["Profit"]
if transactions[transactions['Cout de Revient'] < 0].empty:
    print("Il n'y a pas de transactions avec un coût de revient négatif.")
else:
    print("Il Y a des transactions avec un coût de revient négatif.")

Il n'y a pas de transactions avec un coût de revient négatif.


In [50]:
if transactions[transactions['Sales'] < 0].empty:
    print("Il n'y a pas de transactions avec des ventes négatives.")
else:
    print("Il Y a des transactions avec des ventes négatives.")



Il n'y a pas de transactions avec des ventes négatives.


#### Vérification Logique des Frais de Port 
Il arrive que le coût de livraison (Shipping Cost) soit supérieur au montant total de la commande (Sales). C'est une anomalie logistique grave qu'un contrôleur de gestion doit immédiatement signaler. On fait d'abord le text logistique puis on ajoute toutes les exceptions dans une table . Ensuite on ajoute une colonne motif pour spécifier la raison de l'anomalie . Finalement on exporte les exceptions en EXCEL.

In [ ]:
anomalie_logistique= df["Shipping.Cost"]> df["Sales"]
exceptions = df[anomalie_logistique].copy()
exceptions['Motif de l\'anomalie'] =  "Coût de livraison supérieur au montant de la vente"
df[~anomalie_logistique]
print(f"Nettoyage terminé : {len(exceptions)} anomalie(s) isolée(s) dans le rapport d'exceptions.")

#### Concentration de chiffre d'affaire : Test de la loi 80/20

In [ ]:
total_CA = transactions["Sales"].sum()
CA_par_region = df.groupby("Region")["Sales"].sum().reset_index() # The reset_index() is used to convert the Series back to a DataFrame and copy() is used to avoid SettingWithCopyWarning so when u want to add a new column it wont add new rows
CA_par_region["Part de marché"] = (CA_par_region["Sales"] / total_CA) * 100
CA_par_region.sort_values("Part de marché", ascending=False,inplace=True)
CA_par_region["Part de marché Cumulée"] = CA_par_region["Part de marché"].cumsum().round(2) # cumsum() is used to calculate the cumulative sum of the "Part de marché" column, which gives us the cumulative market share as we go down the sorted list of regions.
top_regions = CA_par_region[CA_par_region['Part de marché Cumulée'] <= 80]['Region'].tolist()
if len(top_regions) > 0:
    regions_texte = ", ".join(top_regions)
    value = "⚠️ Alerte Risque de Concentration : Près de 80% de notre chiffre d'affaires repose uniquement sur ces régions : {regions_texte}."
    print(value)
else:
    print("La première région représente à elle seule plus de 80% du CA !")
CA_par_region["Risque de Concentration"] =value


⚠️ Alerte Risque de Concentration : Près de 80% de notre chiffre d'affaires repose uniquement sur ces régions : {regions_texte}.


#### Exportation des données nettoyées

In [135]:

transactions.to_csv("transactions.csv", index=False)
customers.to_csv("customers.csv", index=False)
products.to_csv("products.csv", index=False)
exceptions.to_excel("exceptions_logistiques.xlsx",index=False)
CA_par_region.to_excel("CA_par_region.xlsx", index=False)
print("Nettoyage et exportation terminés avec succès !")

Nettoyage et exportation terminés avec succès !
